# Expert Q-values with FrozenLake

`q_star_source` is an opt-in field on `EnvConfig` that attaches expert Q-values to every step output as `outputs[i]["q_star"]`. Once configured, the array is available at every step — including the very first reset frame — so a rollout can choose greedy expert actions without training a separate policy.

## Why attach Q-values at the env level?

Attaching Q* at rollout time rather than post-hoc keeps training code clean: the expert is just another field in the output dict, treated the same way as `reward` or `observation`. It is useful for:

- **Imitation and distillation** — supervise a learned policy against `argmax(q_star)` without a separate data pipeline.
- **Diagnostics** — compare the agent's value estimates against exact Q-values to measure how far the learned policy has drifted from optimal.
- **Guided exploration** — epsilon-greedy or softmax over `q_star` to bias rollouts toward useful states without hardcoding a policy.

## `q_star_source` providers

`q_star_source` is a plain dict with a required `"provider"` key. Three providers are available:

### `"env_q_star"` — env-computed Q*

The env runs value iteration internally and injects Q-values into every step output as `info_env_q_star`. No file required. Only supported by `SyntheticEnv-v1` and `Procedural-FrozenLake-v1`.

```python
q_star_source={"provider": "env_q_star"}
```

### `"hf_q_table"` — precomputed tabular Q-table

Load a Q-table from a local pickle file or the Hugging Face Hub. The pickle must contain `{"qtable": np.ndarray[states, actions]}` or be a bare `ndarray`.

| Field | Required | Description |
|---|---|---|
| `"path"` | if no `repo_id` | Local path to a `.pkl` file |
| `"repo_id"` | if no `path` | HF Hub repo ID, e.g. `"user/my-qtable"` |
| `"filename"` | no | File in the Hub repo (default: `"q-learning.pkl"`) |
| `"deterministic"` | no | Argmax action selection (default: `True`) |

### `"sb3_rl_zoo"` — Stable-Baselines3 checkpoint

Load an SB3 policy from a local `.zip` or the Hugging Face Hub. Requires `stable-baselines3`; `"qrdqn"` also requires `sb3-contrib`.

| Field | Required | Description |
|---|---|---|
| `"algo"` | yes | `"a2c"`, `"ddpg"`, `"dqn"`, `"ppo"`, `"sac"`, `"td3"`, `"qrdqn"` |
| `"path"` | if no `repo_id` | Local path to an SB3 `.zip` checkpoint |
| `"repo_id"` | if no `path` | HF Hub repo ID |
| `"filename"` | if no `path` | File in the Hub repo |
| `"device"` | no | `"cpu"`, `"cuda"`, or `"auto"` (default: `"cpu"`) |
| `"deterministic"` | no | Deterministic action selection (default: `True`) |

---

This notebook uses `FrozenLake-v1` (fixed 4×4 map, slippery) with the `hf_q_table` provider. Q-values are computed offline by **value iteration** and saved to a local pickle file, then loaded at env-build time.

## Imports

In [1]:
import pickle
import tempfile

import gymnasium as gym
import numpy as np
import torch

from mouse_envs import EnvConfig, make_env

## Compute Q* via value iteration

`FrozenLake-v1` is a small finite MDP (16 states × 4 actions). Value iteration converges to the exact optimal Q-table in a few hundred iterations. `env.unwrapped.P[s][a]` gives the transition list `(prob, next_state, reward, done)` directly from Gymnasium.

In [2]:
fl = gym.make("FrozenLake-v1", is_slippery=True)
base = fl.unwrapped
n_states  = base.observation_space.n  # 16
n_actions = base.action_space.n       # 4
fl.close()

gamma = 0.99
V = np.zeros(n_states)
for _ in range(1000):
    Q = np.zeros((n_states, n_actions))
    for s in range(n_states):
        for a in range(n_actions):
            for prob, next_s, reward, done in base.P[s][a]:
                Q[s, a] += prob * (reward + gamma * V[next_s] * (not done))
    V = Q.max(axis=1)

print("Q* (first 4 states):")
for s in range(4):
    print(f"  s={s}  {Q[s].round(3)}  greedy_action={int(np.argmax(Q[s]))}")

Q* (first 4 states):
  s=0  [0.542 0.528 0.528 0.522]  greedy_action=0
  s=1  [0.343 0.334 0.32  0.499]  greedy_action=3
  s=2  [0.438 0.434 0.424 0.471]  greedy_action=3
  s=3  [0.306 0.306 0.302 0.457]  greedy_action=3


## Save Q-table and build the environment

The `hf_q_table` provider reads a pickle file containing `{"qtable": np.ndarray[states, actions]}`. Passing a local `path` skips any Hugging Face download.

In [3]:
tmp = tempfile.NamedTemporaryFile(suffix=".pkl", delete=False)
pickle.dump({"qtable": Q}, tmp)
tmp.flush()
qtable_path = tmp.name

cfg = EnvConfig(
    id="FrozenLake-v1",
    seed=0,
    num_envs=1,
    episodes_per_task=100,
    kwargs={"is_slippery": True},
    q_star_source={"provider": "hf_q_table", "path": qtable_path},
)
env = make_env(cfg)

## Inspect the reset frame

The first `step` performs the internal reset. `q_star` is already attached — the 4 Q-values for the start state.

In [4]:
outputs = env.step(env.sample_random_inputs())

print(f"observation (state index): {outputs[0]['observation'].item()}")
print(f"q_star:                    {outputs[0]['info_env_q_star'].round(3)}")
print(f"greedy action:             {int(np.argmax(outputs[0]['info_env_q_star']))}")

observation (state index): 0
q_star:                    [0.542 0.528 0.528 0.522]
greedy action:             0


## Expert rollout

Take `argmax` over `q_star` each step. Because FrozenLake is slippery, the agent does not always move where intended, so episodes may still fail — but the greedy policy is optimal in expectation.

In [5]:
episodes = 0

for step in range(300):
    inputs = [{
        "action": torch.tensor(int(np.argmax(outputs[0]["info_env_q_star"])), dtype=torch.int64)
    }]
    outputs = env.step(inputs)

    r = outputs[0]
    if r["done"].item() != 0:
        episodes += 1
        outcome = "reached goal" if r["reward"].item() > 0 else "fell in hole"
        print(f"step={step:3d}  episode={episodes}  {outcome}")

env.close()

step= 10  episode=1  reached goal
step= 31  episode=2  reached goal
step= 86  episode=3  reached goal
step=167  episode=4  reached goal
step=199  episode=5  fell in hole
step=216  episode=6  reached goal
step=236  episode=7  reached goal
step=255  episode=8  reached goal
step=298  episode=9  reached goal
